# EviChange: compare other models on the 80 held-out boxes (zero-shot)

Runs three other open vision-language models, untrained, on the same 80 held-out boxes and the same prompt as the
4b fine-tune, so they go in one table with the base Qwen2.5-VL-7B (19.4 %) and the fine-tuned model (70.6 %).

| Model | Size | How it loads |
|---|---|---|
| Qwen3-VL-8B-Instruct | 8B | bf16 |
| InternVL3-8B (hf) | 8B | bf16 |
| Qwen2.5-VL-32B-Instruct | 32B | 4-bit |

**Before you start**
- **Runtime → Change runtime type → A100 GPU.**
- Have these 2 files from `D:\EviChange\data\finetune\` ready: `chips.zip` and `cover_holdout.jsonl`.
- Then **Runtime → Run all**. Cell 3 waits for the upload.

It takes about **1–1.5 hours** (mostly downloading the models; the 32B one is about 65 GB) and roughly
**15–20 compute units**. A model that fails is skipped and the others still run.
Cell 5 downloads `compare_results.zip` at the end. **Send it to P2.**

## 1. Install, and check the GPU

In [ ]:
# Colab's old torchao breaks peft imports; it isn't needed. bitsandbytes loads the 32B model in 4-bit.
!pip -q uninstall -y torchao
!pip -q install -U "transformers>=4.57" peft accelerate pillow bitsandbytes

import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 GPU"
name = torch.cuda.get_device_name(0); mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {mem:.0f} GB")
assert mem > 35, "Needs an A100 (40 GB or more)"

## 2. Get the code

In [ ]:
!rm -rf /content/EviChange && git clone -q --depth 1 https://github.com/nachikethpt/EviChange /content/EviChange
!grep -c COMPARE_MODELS /content/EviChange/scripts/finetune_train.py

## 3. Upload the data

Click **Choose files** and pick the **2 files** from `D:\EviChange\data\finetune\` (hold Ctrl):
`chips.zip` and `cover_holdout.jsonl`. Files already uploaded are kept if you run this cell again.

In [ ]:
import os, zipfile
from pathlib import Path
from google.colab import files

FT = Path("/content/ft"); FT.mkdir(exist_ok=True)
os.chdir(FT)
NEEDED = ["chips.zip", "cover_holdout.jsonl"]
missing = [f for f in NEEDED if not (FT / f).exists()]
if missing:
    print("pick:", ", ".join(missing))
    files.upload()
for p in FT.glob("* (*)*"):                      # a re-upload is saved as "name (1).ext"
    clean = FT / (p.stem.split(" (")[0] + p.suffix)
    p.replace(clean)
missing = [f for f in NEEDED if not (FT / f).exists()]
assert not missing, f"still missing: {missing}"
with zipfile.ZipFile(FT / "chips.zip") as z:
    z.extractall(FT)
n = sum(1 for _ in open(FT / "cover_holdout.jsonl", encoding="utf-8"))
assert n == 80, f"cover_holdout.jsonl has {n} boxes, expected 80 (is it the right file?)"
print("data ready: 80 held-out boxes")

## 4. Run each model

One model at a time: download, run on the 80 boxes, score, then delete it to free the disk. Each model's score is
printed as it finishes (main-cover accuracy and the derived change).

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path
sys.path.insert(0, "/content/EviChange/scripts")
from finetune_train import COMPARE_MODELS, model_tag

FT = Path("/content/ft"); LOGS = FT / "compare_logs"; LOGS.mkdir(exist_ok=True)
HF_CACHE = Path.home() / ".cache" / "huggingface" / "hub"
done = {}
for m in COMPARE_MODELS:
    tag = model_tag(m["base"])
    print(f"=== {m['base']} ===", flush=True)
    cmd = [sys.executable, "/content/EviChange/scripts/finetune_train.py", "eval", "--task", "cover",
           "--data", str(FT), "--base-only", "--base", m["base"]] + (["--four_bit"] if m["four_bit"] else [])
    with open(LOGS / f"{tag}.log", "w") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/content/EviChange")
        for line in proc.stdout:
            log.write(line)
            if not line.startswith(("Loading", "Fetching", "Downloading")):
                print(line, end="", flush=True)
        ok = proc.wait() == 0 and (FT / f"predictions_cover_{tag}.jsonl").exists()
    done[m["base"]] = "ok" if ok else "FAILED (see compare_logs)"
    shutil.rmtree(HF_CACHE, ignore_errors=True)             # free the disk for the next model
print()
for k, v in done.items():
    print(f"{k}: {v}")

## 5. Download the results

This downloads `compare_results.zip` (small). **Send it to P2.**

In [ ]:
import shutil
from pathlib import Path
from google.colab import files
FT = Path("/content/ft"); res = Path("/content/compare_results"); res.mkdir(exist_ok=True)
for p in list(FT.glob("predictions_cover_*.jsonl")):
    shutil.copy(p, res / p.name)
shutil.copytree(FT / "compare_logs", res / "compare_logs", dirs_exist_ok=True)
print(sorted(p.name for p in res.iterdir()))
shutil.make_archive("/content/compare_results", "zip", res)
files.download("/content/compare_results.zip")